In [1]:
import sqlite3
import tkinter as tk
from tkinter import messagebox
import datetime
import pandas as pd
from tkinter import filedialog
from datetime import date
import pandas as pd
from tkinter import filedialog
import datetime as _dt
import re as _re
import pandas as pd
from tkinter import filedialog, messagebox

# Notebook 2: Business Logic (The Brain)
This notebook is the "brain" of the application. It does not contain any visual design (no colors, no layouts). 
Instead, it handles:
1. **The Database:** Creating the SQLite database and saving/retrieving patient data.
2. **Data Validation:** Checking if the nurse typed a valid age or forgot to type a name.
3. **The Logic Handlers:** Functions that listen to the buttons, read the text boxes, save the data, and tell the table to refresh.

# database connection
we will creat database SQLite which incloude all infomration about the visitors number and doctor plane.

In [2]:
# connect to database
def get_db_connection():
    conn = sqlite3.connect('clinic.db')
    # allow access using column name
    conn.row_factory = sqlite3.Row
    return conn
# database setup
def setup_database():
    conn = get_db_connection()
    cursor = conn.cursor()
    # patients table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            phone TEXT,
            age INTEGER,
            status TEXT DEFAULT 'Waiting',
            payment_status TEXT DEFAULT 'Not Paid',
            date_added DATE
        )
    ''')
    # history table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS patient_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER,
            name TEXT,
            phone TEXT,
            age INTEGER,
            payment_status TEXT,
            visit_date DATE
        )
    ''')
    # recovery steps table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS recovery_steps (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER,
            step_order INTEGER,
            description TEXT,
            device_name TEXT,
            duration_mins INTEGER
        )
    ''')
    conn.commit()
    conn.close()
# save patient to database
def save_patient_to_db(
        name,
        phone,
        age,
        payment_status="Not Paid"
):
    conn = get_db_connection()
    cursor = conn.cursor()
    # insert patient with status 'Waiting'
    cursor.execute('''
        INSERT INTO patients
        (
            name,
            phone,
            age,
            status,
            payment_status,
            date_added
        )
        VALUES
        (
            ?,
            ?,
            ?,
            'Waiting',
            ?,
            date('now')
        )
    ''',
    (
        name,
        phone,
        age,
        payment_status
    ))
    conn.commit()
    conn.close()
# get waiting patients
def get_all_waiting_patients():
    conn = get_db_connection()
    cursor = conn.cursor()
    # select 'Waiting' patients, oldest first
    cursor.execute('''
        SELECT *
        FROM patients
        WHERE status='Waiting'
        ORDER BY id ASC
    ''')
    patients = cursor.fetchall()
    conn.close()
    return patients
# move patient to history (done button)
def move_patient_to_history(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    # find patient
    cursor.execute('''
        SELECT *
        FROM patients
        WHERE id=?
    ''',
    (patient_id,))
    patient = cursor.fetchone()
    if patient:
        # save to history
        cursor.execute('''
            INSERT INTO patient_history
            (
                patient_id,
                name,
                phone,
                age,
                payment_status,
                visit_date
            )
            VALUES
            (
                ?,
                ?,
                ?,
                ?,
                ?,
                date('now')
            )
        ''',
        (
            patient["id"],
            patient["name"],
            patient["phone"],
            patient["age"],
            patient["payment_status"]
        ))
        # remove from queue
        cursor.execute('''
            DELETE FROM patients
            WHERE id=?
        ''',
        (patient_id,))
    conn.commit()
    conn.close()
# get patient history (for history window)
def get_patient_history():
    conn = get_db_connection()
    cursor = conn.cursor()
    # newest first
    cursor.execute('''
        SELECT *
        FROM patient_history
        ORDER BY id DESC
    ''')
    history = cursor.fetchall()
    conn.close()
    return history
# delete one history item
def delete_history_item(history_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        DELETE FROM patient_history
        WHERE id=?
    ''',
    (history_id,))
    conn.commit()
    conn.close()
# delete all history
def clear_all_history():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        DELETE FROM patient_history
    ''')
    conn.commit()
    conn.close()
# refresh queue table
def refresh_patient_queue(ui):
    tree = ui['patient_tree']
    # remove old rows
    for item in tree.get_children():
        tree.delete(item)
    # get fresh waiting patients
    patients = get_all_waiting_patients()
    # insert each patient as a row
    for patient in patients:
        tree.insert(
            '',
            tk.END,
            values=(
                patient['id'],
                patient['name'],
                patient['age'],
                patient['phone'],
                patient['payment_status']
            )
        )
# add patient button function
def handle_add_patient(ui):
    # get data from form
    name = ui['entry_name'].get().strip()
    phone = ui['entry_phone'].get().strip()
    age_text = ui['entry_age'].get().strip()
    # payment status (fallback if missing)
    if 'payment_status' in ui:
        payment_status = ui['payment_status'].get()
    else:
        payment_status = "Not Paid"
    # check name
    if name == "":
        messagebox.showwarning(
            "Missing Information",
            "Please enter patient's name."
        )
        return
    # check age is a number
    try:
        age = int(age_text)
    except ValueError:
        messagebox.showerror(
            "Invalid Age",
            "Age must be a number."
        )
        return
    try:
        # save data
        save_patient_to_db(
            name,
            phone,
            age,
            payment_status
        )
        # clear form
        ui['entry_name'].delete(0, tk.END)
        ui['entry_phone'].delete(0, tk.END)
        ui['entry_age'].delete(0, tk.END)
        # update queue
        refresh_patient_queue(ui)
        # success message
        messagebox.showinfo(
            "Success",
            f"Patient {name} added to queue."
        )
    except Exception as e:
        messagebox.showerror(
            "Database Error",
            str(e)
        )
# move patient to history (duplicate — kept as-is)
def move_patient_to_history(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    # find patient
    cursor.execute(
        """
        SELECT *
        FROM patients
        WHERE id=?
        """,
        (patient_id,)
    )
    patient = cursor.fetchone()
    if patient:
        # save into history
        cursor.execute(
        """
        INSERT INTO patient_history
        (
        patient_id,
        name,
        phone,
        age,
        payment_status,
        visit_date
        )
        VALUES (?, ?, ?, ?, ?, date('now'))
        """,
        (
        patient['id'],
        patient['name'],
        patient['phone'],
        patient['age'],
        patient['payment_status']
        )
        )
        # remove from waiting queue
        cursor.execute(
            """
            DELETE FROM patients
            WHERE id=?
            """,
            (patient_id,)
        )
    conn.commit()
    conn.close()
# start database
setup_database()

# In this cell, we write the function that runs when the doctor clicks "Create Recovery Plan". 
The logic checks if a patient is selected in the queue table. 
If no patient is selected, it shows a warning. If a patient is selected, it confirms which patient the doctor is working on.

In [3]:
def handle_create_plan(ui):
    
    # get patient queue table from ui
    tree = ui['patient_tree']
    
    # get selected row id
    selected_items = tree.selection()
    
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return # stop here
    
    # get data from first selected row
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    
    # extract patient id + name
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    
    # confirmation pop-up
    messagebox.showinfo("Doctor Mode Activated", f"Opening Recovery Plan for:\n\nPatient: {patient_name}\nID: {patient_id}\n\n(Next step: We will build the form to add the treatment steps!)")

# handle the logic when doctor press"create recovery plan" button 


In [4]:
def handle_create_plan(ui, root_window):
    
    # get patient queue table from ui
    tree = ui['patient_tree']
    
    # get currently selected row
    selected_items = tree.selection()
    
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    
    # get data from selected row
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    
    # extract patient id + name
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    
    # open recovery plan pop-up
    open_recovery_plan_window(root_window, patient_id, patient_name)

### In this cell, we write the functions that handle adding steps to the temporary list
inside the pop-up window, and saving the complete plan into the SQLite database
when the doctor clicks "Save".

In [5]:
# handle "add step to list" click (inside pop-up)
def handle_add_step_to_list(plan_ui):
    
    # read three input boxes from pop-up
    desc = plan_ui['entry_desc'].get().strip()
    device = plan_ui['entry_device'].get().strip()
    dur_str = plan_ui['entry_duration'].get().strip()
    
    # warn if description or duration missing
    if not desc or not dur_str:
        messagebox.showwarning("Missing Info", "Please fill in the description and duration.")
        return
    
    # validate duration is a number
    try:
        duration = int(dur_str)
    except ValueError:
        messagebox.showerror("Invalid Number", "Duration must be a number (e.g., 15).")
        return
    
    # calculate next step number (step 1, 2, 3...)
    current_items = plan_ui['tree_steps'].get_children()
    next_step = len(current_items) + 1
    
    # add new step as row in pop-up table
    plan_ui['tree_steps'].insert('', tk.END, values=(next_step, desc, device, duration))
    
    # clear entries (ready for next step)
    plan_ui['entry_desc'].delete(0, tk.END)
    plan_ui['entry_device'].delete(0, tk.END)
    plan_ui['entry_duration'].delete(0, tk.END)

# handle "save plan to database" click
def handle_save_plan_to_db(plan_ui, patient_id):
    
    # get steps table from pop-up
    tree = plan_ui['tree_steps']
    items = tree.get_children()
    
    # warn if plan has no steps
    if not items:
        messagebox.showwarning("Empty Plan", "Please add at least one step before saving.")
        return
    
    # connect to database
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # save each step to database
    for item in items:
        values = tree.item(item)['values']
        # values = (step_order, description, device_name, duration_mins)
        cursor.execute('''
            INSERT INTO recovery_steps (patient_id, step_order, description, device_name, duration_mins)
            VALUES (?, ?, ?, ?, ?)
        ''', (patient_id, values[0], values[1], values[2], values[3]))
    
    # commit + close connection
    conn.commit()
    conn.close()
    
    # success message
    messagebox.showinfo("Success", "Recovery plan saved successfully to the database!")
    
    # close pop-up after saving
    plan_ui['window'].destroy()

## build creat recovery pop-up window and connect its buttons

In [6]:
# build pop-up window and connect its buttons
def open_recovery_plan_window(parent, patient_id, patient_name):
    
    # build pop-up
    plan_ui = build_recovery_plan_window(parent, patient_name)
    
    # connect "add step" button
    plan_ui['btn_add_step'].config(command=lambda: handle_add_step_to_list(plan_ui))
    
    # connect "save plan" button
    plan_ui['btn_save_plan'].config(command=lambda: handle_save_plan_to_db(plan_ui, patient_id))

# In this cell, we write the functions that allow the nurse to view the saved steps: 
check them off one by one, and finally update the database to mark the patient as "Done", 
which automatically removes them from the waiting queue.

In [7]:
# get recovery steps for a patient
def get_recovery_steps_for_patient(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    # select steps ordered by step number
    cursor.execute(
        """
        SELECT *
        FROM recovery_steps
        WHERE patient_id = ?
        ORDER BY step_order ASC
        """,
        (patient_id,)
    )
    steps = cursor.fetchall()
    conn.close()
    return steps
# view execution plan button
def handle_view_execution_plan(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning(
            "No Patient Selected",
            "Please click on a patient first."
        )
        return
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    open_execute_plan_window(
        root_window,
        patient_id,
        patient_name,
        ui
    )
# mark step done
def handle_mark_step_done(execute_ui):
    tree = execute_ui['tree_steps']
    selected = tree.selection()
    # warn + stop if no step selected
    if not selected:
        messagebox.showwarning(
            "No Step Selected",
            "Please select a step."
        )
        return
    item = selected[0]
    values = tree.item(item,'values')
    # update status column to done
    tree.item(
        item,
        values=(
            values[0],
            values[1],
            values[2],
            values[3],
            "✅ Done"
        )
    )
# move completed patient to history
def move_patient_to_history(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    # get patient data
    cursor.execute(
        """
        SELECT *
        FROM patients
        WHERE id=?
        """,
        (patient_id,)
    )
    patient = cursor.fetchone()
    if patient:
        # save patient history
        cursor.execute(
        """
        INSERT INTO patient_history
        (
        patient_id,
        name,
        phone,
        age,
        payment_status,
        visit_date
        )
        VALUES
        (
        ?,
        ?,
        ?,
        ?,
        ?,
        date('now')
        )
        """,
        (
        patient['id'],
        patient['name'],
        patient['phone'],
        patient['age'],
        patient['payment_status']
        )
        )
        # remove from queue
        cursor.execute(
        """
        DELETE FROM patients
        WHERE id=?
        """,
        (patient_id,)
        )
    conn.commit()
    conn.close()
# complete patient button
def handle_complete_patient(execute_ui, main_ui):
    patient_id = execute_ui['patient_id']
    try:
        # 1. save patient into history
        move_patient_to_history(patient_id)
        # 2. refresh queue
        refresh_patient_queue(main_ui)
        # 3. success message
        messagebox.showinfo(
            "Completed",
            "Patient visit completed and saved into history."
        )
        # 4. close window
        execute_ui['window'].destroy()
    except Exception as e:
        messagebox.showerror(
            "Error",
            str(e)
        )
# open execution window
def open_execute_plan_window(parent, patient_id, patient_name, main_ui):
    execute_ui = build_execute_plan_window(
        parent,
        patient_name,
        patient_id
    )
    # fetch steps from database
    steps = get_recovery_steps_for_patient(patient_id)
    # no plan → show message + close
    if not steps:
        messagebox.showinfo(
            "No Plan",
            "This patient has no recovery plan."
        )
        execute_ui['window'].destroy()
        return
    # load steps with pending status
    for s in steps:
        execute_ui['tree_steps'].insert(
            '',
            tk.END,
            values=(
                s['step_order'],
                s['description'],
                s['device_name'],
                s['duration_mins'],
                "⏳ Pending"
            )
        )
    # connect mark step button
    execute_ui['btn_mark_step'].config(
        command=lambda:
        handle_mark_step_done(execute_ui)
    )
    # connect complete button
    execute_ui['btn_complete'].config(
        command=lambda:
        handle_complete_patient(
            execute_ui,
            main_ui
        )
    )
# history window button
def handle_view_patient_history(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning(
            "No Patient Selected",
            "Please select patient first."
        )
        return
    item = selected_items[0]
    data = tree.item(item)
    patient_id = data['values'][0]
    patient_name = data['values'][1]
    open_patient_history_window(
        root_window,
        patient_id,
        patient_name
    )
# patient history window
def open_patient_history_window(parent, patient_id, patient_name):
    history_ui = build_patient_history_window(
        parent,
        patient_name,
        patient_id
    )
    # display visit count
    visit_count = get_patient_visit_count(patient_id)
    history_ui['visit_count_label'].config(
        text=f"Total Visits: {visit_count}"
    )
    # get recovery steps
    steps = get_recovery_steps_for_patient(patient_id)
    # populate table
    for s in steps:
        history_ui['tree_history'].insert(
            '',
            tk.END,
            values=(
                s['step_order'],
                s['description'],
                s['device_name'],
                s['duration_mins']
            )
        )
    # connect close button
    history_ui['btn_close'].config(
        command=history_ui['window'].destroy
    )
# visit count functions
def add_visit_count_column():
    conn = get_db_connection()
    cursor = conn.cursor()
    # check existing columns
    cursor.execute(
        "PRAGMA table_info(patients)"
    )
    columns = [
        c[1]
        for c in cursor.fetchall()
    ]
    # add visit_count if missing
    if "visit_count" not in columns:
        cursor.execute(
        """
        ALTER TABLE patients
        ADD COLUMN visit_count INTEGER DEFAULT 0
        """
        )
        conn.commit()
    conn.close()
# increment visit count
def increment_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        UPDATE patients
        SET visit_count = visit_count + 1
        WHERE id=?
        """,
        (patient_id,)
    )
    conn.commit()
    conn.close()
# get visit count
def get_patient_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT visit_count
        FROM patients
        WHERE id=?
        """,
        (patient_id,)
    )
    result = cursor.fetchone()
    conn.close()
    # return count, or 0 if not found
    if result:
        return result['visit_count']
    return 0
# initialize
add_visit_count_column()

# show completed message upon recording 
the visit enhanced mark-done (logs visits + audit trail)

In [8]:
def handle_complete_patient_v2(execute_ui, main_ui):
    """enhanced version: logs visits and audit trail"""
    patient_id = execute_ui['patient_id']
    
    # update patient status to 'done'
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE patients SET status = 'Done' WHERE id = ?", (patient_id,))
    conn.commit()
    conn.close()
    
    # log the visit
    log_visit(patient_id, "Done", "Treatment completed", "Nurse")
    
    # log to audit trail
    log_audit("UPDATE", "patients", patient_id, "Waiting", "Done", "Nurse")
    
    # success message with remaining visits
    remaining = get_remaining_visits(patient_id)
    messagebox.showinfo("Success", f"Patient marked as Done!\n\nRemaining visits: {remaining}")
    
    # refresh main queue
    refresh_patient_queue(main_ui)
    
    # close execution window
    execute_ui['window'].destroy()

# In this cell, we write the function that define the delete button. 
It checks if a patient is selected, asks the nurse for confirmation to prevent accidents, 
and then permanently removes the patient (and their recovery steps) from the SQLite database.

In [9]:
def delete_patient_from_db(patient_id):
    # connect to database
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # 1. delete associated recovery steps (keeps database clean)
    cursor.execute("DELETE FROM recovery_steps WHERE patient_id = ?", (patient_id,))
    
    # 2. delete patient record
    cursor.execute("DELETE FROM patients WHERE id = ?", (patient_id,))
    
    # save changes
    conn.commit()
    # close connection
    conn.close()

# handle red "delete selected patient" click
def handle_delete_patient(ui):
    
    # get patient queue table from ui
    tree = ui['patient_tree']
    
    # get currently selected row id
    selected_items = tree.selection()
    
    # warn and stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue to delete them.")
        return # stop here
    
    # get data from first selected row
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    
    # extract patient id + name
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    
    # confirmation pop-up
    confirm = messagebox.askyesno("Confirm Deletion", f"Are you sure you want to permanently delete:\n\n{patient_name}\n\nThis action cannot be undone.")
    
    # stop if nurse clicked "no"
    if not confirm:
        return
        
    # delete patient from database
    delete_patient_from_db(patient_id)
    
    # refresh queue
    refresh_patient_queue(ui)
    
    # success message
    messagebox.showinfo("Success", f"Patient {patient_name} has been removed from the queue.")

# add history button logic
and add visit_count column if missing

In [10]:
def add_visit_count_column():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # check existing columns
    cursor.execute("PRAGMA table_info(patients)")
    columns = [column[1] for column in cursor.fetchall()]
    
    # add visit_count if missing
    if 'visit_count' not in columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN visit_count INTEGER DEFAULT 0")
        conn.commit()
    
    conn.close()

# increment visit count on patient completion
def increment_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE patients SET visit_count = visit_count + 1 WHERE id = ?", (patient_id,))
    conn.commit()
    conn.close()

# get patient visit count
def get_patient_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT visit_count FROM patients WHERE id = ?", (patient_id,))
    result = cursor.fetchone()
    conn.close()
    return result['visit_count'] if result else 0

# handle patient history click
def handle_view_patient_history(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    
    open_patient_history_window(root_window, patient_id, patient_name)

# open + populate history window
def open_patient_history_window(parent, patient_id, patient_name):
    history_ui = build_patient_history_window(parent, patient_name, patient_id)
    
    # display visit count
    visit_count = get_patient_visit_count(patient_id)
    history_ui['visit_count_label'].config(text=f"Total Visits: {visit_count}")
    
    # get all recovery steps
    steps = get_recovery_steps_for_patient(patient_id)
    
    # populate table
    for s in steps:
        history_ui['tree_history'].insert('', tk.END, values=(
            s['step_order'], 
            s['description'], 
            s['device_name'], 
            s['duration_mins']
        ))
    
    # connect close button
    history_ui['btn_close'].config(command=history_ui['window'].destroy)

# initialize visit_count column on load
add_visit_count_column()

# in this cell we will add visit_count column if missing


In [11]:
def add_visit_count_column():
    conn = get_db_connection()
    cursor = conn.cursor()
    # check existing columns
    cursor.execute("PRAGMA table_info(patients)")
    columns = [column[1] for column in cursor.fetchall()]
    # add visit_count if missing
    if 'visit_count' not in columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN visit_count INTEGER DEFAULT 0")
        conn.commit()
    conn.close()

# increment visit count
def increment_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE patients SET visit_count = visit_count + 1 WHERE id = ?", (patient_id,))
    conn.commit()
    conn.close()

# get visit count
def get_patient_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT visit_count FROM patients WHERE id = ?", (patient_id,))
    result = cursor.fetchone()
    conn.close()
    return result['visit_count'] if result else 0

# handle patient history click
def handle_view_patient_history(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    open_patient_history_window(root_window, patient_id, patient_name)

# open + populate history window
def open_patient_history_window(parent, patient_id, patient_name):
    history_ui = build_patient_history_window(parent, patient_name, patient_id)
    # display visit count
    visit_count = get_patient_visit_count(patient_id)
    history_ui['visit_count_label'].config(text=f"Total Visits: {visit_count}")
    # get recovery steps
    steps = get_recovery_steps_for_patient(patient_id)
    # populate table
    for s in steps:
        history_ui['tree_history'].insert('', tk.END, values=(
            s['step_order'], 
            s['description'], 
            s['device_name'], 
            s['duration_mins']
        ))
    # connect close button
    history_ui['btn_close'].config(command=history_ui['window'].destroy)

# initialize visit_count column on load
add_visit_count_column()

# handle patient history click


In [12]:
def handle_view_patient_history(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    open_patient_history_window(root_window, patient_id, patient_name)

# open + populate history window
def open_patient_history_window(parent, patient_id, patient_name):
    history_ui = build_patient_history_window(parent, patient_name, patient_id)
    # display visit count
    visit_count = get_patient_visit_count(patient_id)
    history_ui['visit_count_label'].config(text=f"Total Visits: {visit_count}")
    # get recovery steps
    steps = get_recovery_steps_for_patient(patient_id)
    # populate table
    for s in steps:
        history_ui['tree_history'].insert('', tk.END, values=(s['step_order'], s['description'], s['device_name'], s['duration_mins']))
    # connect close button
    history_ui['btn_close'].config(command=history_ui['window'].destroy)

# Patient History Viewer Functions
In this cell, we define three functions. fetches the visit count from SQLite. checks if a patient is selected, warns if not, and opens the history window.  shows the count, fills the table with recovery steps, and wires the close button.

In [13]:
# get visit count 
def get_patient_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT visit_count FROM patients WHERE id = ?", (patient_id,))
    result = cursor.fetchone()
    conn.close()
    # return count, or 0 if not found
    if result:
        return result['visit_count']
    return 0

# handle patient history click
def handle_view_patient_history(ui, root_window):
    tree = ui['patient_tree']
    selected_items = tree.selection()
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    open_patient_history_window(root_window, patient_id, patient_name)

# open + populate history window
def open_patient_history_window(parent, patient_id, patient_name):
    history_ui = build_patient_history_window(parent, patient_name, patient_id)
    # display visit count
    visit_count = get_patient_visit_count(patient_id)
    history_ui['visit_count_label'].config(text=f"Total Visits: {visit_count}")
    # get recovery steps
    steps = get_recovery_steps_for_patient(patient_id)
    # populate table
    for s in steps:
        history_ui['tree_history'].insert('', tk.END, values=(s['step_order'], s['description'], s['device_name'], s['duration_mins']))
    # connect close button
    history_ui['btn_close'].config(command=history_ui['window'].destroy)

# in this cell we will evolve database
 to incloude new advantage

In [14]:
def upgrade_database_schema():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # check existing columns in patients table
    cursor.execute("PRAGMA table_info(patients)")
    patient_columns = [column[1] for column in cursor.fetchall()]
    
    # add total_prescribed_visits column if missing
    if 'total_prescribed_visits' not in patient_columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN total_prescribed_visits INTEGER DEFAULT 0")
    
    # create visit_log table to track each visit
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS visit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER,
            visit_date TEXT,
            status TEXT,
            notes TEXT,
            created_by TEXT,
            created_at TEXT,
            FOREIGN KEY (patient_id) REFERENCES patients (id)
        )
    ''')
    
    # create audit_log table to track all changes
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            action_type TEXT,
            table_name TEXT,
            record_id INTEGER,
            old_value TEXT,
            new_value TEXT,
            performed_by TEXT,
            performed_at TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print("Database schema upgraded successfully!")

# run the upgrade
upgrade_database_schema()

Database schema upgraded successfully!


# devolpe visit tracking functions
by  log a visit to visit_log table

In [15]:
def log_visit(patient_id, status, notes="", created_by="Nurse"):
    """log a visit to the visit_log table"""
    conn = get_db_connection()
    cursor = conn.cursor()
    visit_date = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute('''
        INSERT INTO visit_log (patient_id, visit_date, status, notes, created_by, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (patient_id, visit_date, status, notes, created_by, visit_date))
    conn.commit()
    conn.close()

# get all visits for a patient
def get_visit_history(patient_id):
    """get all visits for a patient"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT * FROM visit_log 
        WHERE patient_id = ? 
        ORDER BY visit_date DESC
    ''', (patient_id,))
    visits = cursor.fetchall()
    conn.close()
    return visits

# get total number of visits
def get_visit_count(patient_id):
    """get total number of visits"""
    visits = get_visit_history(patient_id)
    return len(visits)

# get number of completed visits
def get_completed_visit_count(patient_id):
    """get number of completed visits"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT COUNT(*) as count FROM visit_log 
        WHERE patient_id = ? AND status = 'Done'
    ''', (patient_id,))
    result = cursor.fetchone()
    conn.close()
    return result['count'] if result else 0

# set how many visits the doctor prescribed
def set_total_prescribed_visits(patient_id, total_visits):
    """set how many visits the doctor prescribed"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        UPDATE patients SET total_prescribed_visits = ? WHERE id = ?
    ''', (total_visits, patient_id))
    conn.commit()
    conn.close()

# get total prescribed visits
def get_total_prescribed_visits(patient_id):
    """get total prescribed visits"""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        SELECT total_prescribed_visits FROM patients WHERE id = ?
    ''', (patient_id,))
    result = cursor.fetchone()
    conn.close()
    return result['total_prescribed_visits'] if result else 0

# calculate remaining visits
def get_remaining_visits(patient_id):
    """calculate remaining visits"""
    total = get_total_prescribed_visits(patient_id)
    completed = get_completed_visit_count(patient_id)
    remaining = total - completed
    return max(0, remaining)  # never show negative

# This cell includes two functions for managing the audit trail:

The first opens a connection to the SQLite database and inserts a new row into the `audit_log` table containing the action type, table name, record ID, old and new values ​​(converted to strings), the actor (defaulting to "System"), and the timestamp; it then commits the changes and closes the connection.
The second opens a connection and retrieves records; if a record ID and table name are specified, the results are filtered to include only that specific record and table, displaying the most recent entries first. If they are not specified, the function returns the 100 most recent records, closes the connection, and returns the list.

In [16]:
def log_audit(action_type, table_name, record_id, old_value, new_value, performed_by="System"):
    """log an action to the audit trail"""
    conn = get_db_connection()
    cursor = conn.cursor()
    performed_at = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    cursor.execute('''
        INSERT INTO audit_log (action_type, table_name, record_id, old_value, new_value, performed_by, performed_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (action_type, table_name, record_id, str(old_value), str(new_value), performed_by, performed_at))
    conn.commit()
    conn.close()

# get audit log entries
def get_audit_log(record_id=None, table_name=None):
    """get audit log entries"""
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # filter by record + table if both given
    if record_id and table_name:
        cursor.execute('''
            SELECT * FROM audit_log 
            WHERE record_id = ? AND table_name = ?
            ORDER BY performed_at DESC
        ''', (record_id, table_name))
    else:
        cursor.execute('SELECT * FROM audit_log ORDER BY performed_at DESC LIMIT 100')
    
    logs = cursor.fetchall()
    conn.close()
    return logs

# enhanced history window functions

Four functions: view history, build window, edit visit total, undo last visit.

In [17]:
# open the enhanced history window
def handle_view_enhanced_history(ui, root_window):
    """open the enhanced history window"""
    tree = ui['patient_tree']
    selected_items = tree.selection()
    
    # warn + stop if no patient selected
    if not selected_items:
        messagebox.showwarning("No Patient Selected", "Please click on a patient in the Waiting Queue first.")
        return
    
    selected_item = selected_items[0]
    item_data = tree.item(selected_item)
    patient_id = item_data['values'][0]
    patient_name = item_data['values'][1]
    
    open_enhanced_history_window(root_window, patient_id, patient_name)

# open + populate enhanced history window
def open_enhanced_history_window(parent, patient_id, patient_name):
    """open and populate the enhanced history window"""
    history_ui = build_enhanced_history_window(parent, patient_name, patient_id)
    
    # load visit history
    visits = get_visit_history(patient_id)
    for v in visits:
        history_ui['tree_visits'].insert('', tk.END, values=(
            v['visit_date'],
            v['status'],
            v['notes'],
            v['created_by']
        ))
    
    # connect buttons
    history_ui['btn_close'].config(command=history_ui['window'].destroy)
    history_ui['btn_edit_visits'].config(command=lambda: handle_edit_prescribed_visits(history_ui))
    history_ui['btn_undo'].config(command=lambda: handle_undo_last_visit(history_ui))

# open dialog to edit prescribed visits
def handle_edit_prescribed_visits(history_ui):
    """open dialog to edit prescribed visits"""
    patient_id = history_ui['patient_id']
    current_total = get_total_prescribed_visits(patient_id)
    
    # simple dialog
    dialog = tk.Toplevel(history_ui['window'])
    dialog.title("Edit Prescribed Visits")
    dialog.geometry("300x150")
    dialog.configure(bg=COLOR_BG)
    
    tk.Label(dialog, text=f"Current: {current_total} visits", 
             font=("Arial", 12), bg=COLOR_BG, fg=COLOR_TEXT).pack(pady=10)
    
    tk.Label(dialog, text="New Total:", font=("Arial", 11), bg=COLOR_BG, fg=COLOR_TEXT).pack()
    entry_new_total = tk.Entry(dialog, font=("Arial", 12))
    entry_new_total.pack(pady=5)
    entry_new_total.insert(0, str(current_total))
    
    # save new total
    def save_new_total():
        try:
            new_total = int(entry_new_total.get())
            old_total = get_total_prescribed_visits(patient_id)
            set_total_prescribed_visits(patient_id, new_total)
            log_audit("UPDATE", "patients", patient_id, f"total_prescribed={old_total}", 
                     f"total_prescribed={new_total}", "Doctor")
            messagebox.showinfo("Success", f"Updated to {new_total} total visits")
            dialog.destroy()
            history_ui['window'].destroy()
            # refresh history window
            open_enhanced_history_window(history_ui['window'].master, patient_id, history_ui['patient_name'])
        except ValueError:
            messagebox.showerror("Error", "Please enter a valid number")
    
    tk.Button(dialog, text="Save", command=save_new_total, 
             bg=COLOR_PRIMARY, fg=COLOR_BTN_TEXT).pack(pady=10)

# undo the last visit
def handle_undo_last_visit(history_ui):
    """undo the last visit"""
    patient_id = history_ui['patient_id']
    
    # confirmation pop-up
    confirm = messagebox.askyesno("Confirm Undo", 
                                  "Are you sure you want to undo the last visit?\n\nThis will remove the most recent visit record.")
    
    if not confirm:
        return
    
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # get the last visit
    cursor.execute('''
        SELECT id FROM visit_log 
        WHERE patient_id = ? 
        ORDER BY visit_date DESC 
        LIMIT 1
    ''', (patient_id,))
    last_visit = cursor.fetchone()
    
    if last_visit:
        visit_id = last_visit['id']
        cursor.execute("DELETE FROM visit_log WHERE id = ?", (visit_id,))
        
        # if visit was 'done', revert status to 'waiting'
        cursor.execute('''
            UPDATE patients SET status = 'Waiting' WHERE id = ?
        ''', (patient_id,))
        
        conn.commit()
        
        # log the undo action
        log_audit("UNDO", "visit_log", visit_id, "Deleted", "Undone", "Nurse")
    
    conn.close()
    
    messagebox.showinfo("Success", "Last visit undone")
    history_ui['window'].destroy()
    # refresh history window
    open_enhanced_history_window(history_ui['window'].master, patient_id, history_ui['patient_name'])

# recovery plan save functions

two functions save and open the plan window. The save handler warns if empty, validates the visit total, saves it to the patients table, logs it, inserts each step into recovery_steps, then confirms and closes the window. The opener builds the window and wires the add step and save plan buttons.

In [18]:
# save plan + set prescribed visits
def handle_save_plan_to_db_v2(plan_ui, patient_id):
    """save plan and set prescribed visits"""
    tree = plan_ui['tree_steps']
    items = tree.get_children()
    
    # warn + stop if plan has no steps
    if not items:
        messagebox.showwarning("Empty Plan", "Please add at least one step before saving.")
        return
    
    # get + validate total prescribed visits
    try:
        total_visits = int(plan_ui['entry_total_visits'].get())
        if total_visits < 1:
            raise ValueError
    except ValueError:
        messagebox.showerror("Invalid Number", "Total prescribed visits must be a positive number.")
        return
    
    # save prescribed visits
    set_total_prescribed_visits(patient_id, total_visits)
    log_audit("UPDATE", "patients", patient_id, "total_prescribed_visits", 
             str(total_visits), "Doctor")
    
    # save recovery steps
    conn = get_db_connection()
    cursor = conn.cursor()
    for item in items:
        values = tree.item(item)['values']
        cursor.execute('''
            INSERT INTO recovery_steps (patient_id, step_order, description, device_name, duration_mins)
            VALUES (?, ?, ?, ?, ?)
        ''', (patient_id, values[0], values[1], values[2], values[3]))
    conn.commit()
    conn.close()
    
    # success message
    messagebox.showinfo("Success", f"Recovery plan saved!\n\nTotal prescribed visits: {total_visits}")
    plan_ui['window'].destroy()

# open recovery plan window v2
def open_recovery_plan_window_v2(parent, patient_id, patient_name):
    """open recovery plan window v2"""
    plan_ui = build_recovery_plan_window_v2(parent, patient_name)
    # connect add step button
    plan_ui['btn_add_step'].config(command=lambda: handle_add_step_to_list(plan_ui))
    # connect save plan button
    plan_ui['btn_save_plan'].config(command=lambda: handle_save_plan_to_db_v2(plan_ui, patient_id))

# this cell sets up Excel export and adds database columns. 


In [19]:
# pandas for excel files, filedialog for save location
import pandas as pd
from tkinter import filedialog
from datetime import date

# add 'date_added' + 'payment_status' columns if missing
def add_excel_columns():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # get all current columns in patients table
    cursor.execute("PRAGMA table_info(patients)")
    columns = [column[1] for column in cursor.fetchall()]
    
    # add 'date_added' if missing (no default — set on insert)
    if 'date_added' not in columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN date_added TEXT")
        
    # add 'payment_status' if missing (default 'Unpaid')
    if 'payment_status' not in columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN payment_status TEXT DEFAULT 'Unpaid'")
        
    conn.commit()
    conn.close()

# handle "export to excel" button click
def handle_export_to_excel():
    conn = get_db_connection()
    
    # read all patient data into a dataframe
    df = pd.read_sql_query("SELECT id, name, phone, age, visit_count, payment_status, date_added, status FROM patients", conn)
    conn.close()
    
    # warn + stop if database is empty
    if df.empty:
        messagebox.showinfo("No Data", "There are no patients in the database to export.")
        return
        
    # ask user where to save the excel file
    file_path = filedialog.asksaveasfilename(
        defaultextension=".xlsx",
        filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")],
        title="Save Patient Report"
    )
    
    # stop if user clicked cancel
    if not file_path:
        return 
        
    try:
        # save data to excel file
        df.to_excel(file_path, index=False, sheet_name="Patient Data")
        # success message with file path
        messagebox.showinfo("Success", f"Report successfully saved to:\n{file_path}")
    except Exception as e:
        # error (e.g. file open in excel)
        messagebox.showerror("Error", f"Failed to save file. Make sure the Excel file is closed!\n\nError: {e}")

# run column check on load
add_excel_columns()

# fix cell that show  and run old broken versions


In [20]:
# correct save function: 4 values match 4 placeholders
def save_patient_to_db(name, phone, age, payment_status):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO patients (name, phone, age, status, payment_status, date_added)
        VALUES (?, ?, ?, 'Waiting', ?, date('now'))
    ''', (name, phone, age, payment_status))
    conn.commit()
    conn.close()

# check existing patient
def check_existing_patient(name, phone):
    conn = get_db_connection()
    cursor = conn.cursor()
    # search by name or phone
    cursor.execute(
        """
        SELECT *
        FROM patients
        WHERE name = ?
        OR phone = ?
        """,
        (
            name,
            phone
        )
    )
    patient = cursor.fetchone()
    conn.close()
    return patient

# get patient visit history
def get_patient_previous_visits(name):
    conn = get_db_connection()
    cursor = conn.cursor()
    # select history for this name, newest first
    cursor.execute(
        """
        SELECT *
        FROM patient_history
        WHERE name=?
        ORDER BY id DESC
        """,
        (name,)
    )
    visits = cursor.fetchall()
    conn.close()
    return visits

# add patient button
def handle_add_patient(ui):
    # get data from form
    name = ui['entry_name'].get().strip()
    phone = ui['entry_phone'].get().strip()
    age_text = ui['entry_age'].get().strip()
    
    # payment status (fallback if missing)
    if 'payment_status' in ui:
        payment_status = ui['payment_status'].get()
    else:
        payment_status = "Not Paid"
    
    # check empty name
    if name == "":
        messagebox.showwarning(
            "Missing Information",
            "Please enter patient's name."
        )
        return
    
    # check age is a number
    try:
        age = int(age_text)
    except ValueError:
        messagebox.showerror(
            "Invalid Age",
            "Age must be a number."
        )
        return
    
    # check duplicate patient
    existing = check_existing_patient(
        name,
        phone
    )
    
    if existing:
        # get previous visits for this patient
        visits = get_patient_previous_visits(
            existing['name']
        )
        # build warning message
        message = f"""
Patient already exists!

Name:
{existing['name']}

Phone:
{existing['phone']}

Age:
{existing['age']}

Previous Visits:
{len(visits)}
        """
        messagebox.showwarning(
            "Existing Patient",
            message
        )
        return
    
    # save new patient
    try:
        save_patient_to_db(
            name,
            phone,
            age,
            payment_status
        )
        # clear form
        ui['entry_name'].delete(0,tk.END)
        ui['entry_phone'].delete(0,tk.END)
        ui['entry_age'].delete(0,tk.END)
        # update queue
        refresh_patient_queue(ui)
        # success message
        messagebox.showinfo(
            "Success",
            f"Patient {name} added."
        )
    except Exception as e:
        messagebox.showerror(
            "Database Error",
            str(e)
        )

# add visit_count column if missing
def add_visit_count_column():
    conn = get_db_connection()
    cursor = conn.cursor()
    # check existing columns
    cursor.execute("PRAGMA table_info(patients)")
    columns = [column[1] for column in cursor.fetchall()]
    # add visit_count if missing
    if 'visit_count' not in columns:
        cursor.execute("ALTER TABLE patients ADD COLUMN visit_count INTEGER DEFAULT 0")
        conn.commit()
    conn.close()

# increment visit count
def increment_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE patients SET visit_count = visit_count + 1 WHERE id = ?", (patient_id,))
    conn.commit()
    conn.close()

# get visit count
def get_patient_visit_count(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT visit_count FROM patients WHERE id = ?", (patient_id,))
    result = cursor.fetchone()
    conn.close()
    return result['visit_count'] if result else 0

# correct complete handler: now increases visit count
def handle_complete_patient(execute_ui, main_ui):
    patient_id = execute_ui['patient_id']
    # update status to 'done'
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE patients SET status = 'Done' WHERE id = ?", (patient_id,))
    conn.commit()
    conn.close()
    # increment visit count
    increment_visit_count(patient_id)
    messagebox.showinfo("Success", "Patient marked as Done! Moving to the next patient.")
    refresh_patient_queue(main_ui)
    execute_ui['window'].destroy()

# excel export handler
def handle_export_to_excel():
    conn = get_db_connection()
    df = pd.read_sql_query("SELECT id, name, phone, age, visit_count, payment_status, date_added, status FROM patients", conn)
    conn.close()
    # warn if no data
    if df.empty:
        messagebox.showinfo("No Data", "There are no patients in the database to export.")
        return
    # ask where to save
    file_path = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel files", "*.xlsx")], title="Save Patient Report")
    if not file_path:
        return
    # save + handle errors
    try:
        df.to_excel(file_path, index=False, sheet_name="Patient Data")
        messagebox.showinfo("Success", f"Report saved to:\n{file_path}")
    except Exception as e:
        messagebox.showerror("Error", f"Failed to save. Close the Excel file first!\n{e}")

# ensure visit_count column exists
add_visit_count_column()

# in this cell we will add the final patch
visitor history system and display this cell overrides old history

In [21]:
# move completed patient to history
def move_patient_to_history(patient_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    # find patient
    cursor.execute("SELECT * FROM patients WHERE id=?", (patient_id,))
    patient = cursor.fetchone()

    if patient:
        # save to history
        cursor.execute("""
            INSERT INTO patient_history
            (patient_id, name, phone, age, payment_status, visit_date)
            VALUES (?, ?, ?, ?, ?, date('now'))
        """, (
            patient["id"],
            patient["name"],
            patient["phone"],
            patient["age"],
            patient["payment_status"]
        ))

        # remove from queue
        cursor.execute("DELETE FROM patients WHERE id=?", (patient_id,))

    conn.commit()
    conn.close()

# complete patient button
def handle_complete_patient(execute_ui, main_ui):
    try:
        # save to history + refresh queue
        move_patient_to_history(execute_ui["patient_id"])
        refresh_patient_queue(main_ui)

        # success message
        messagebox.showinfo(
            "Completed",
            "Patient saved into visitor history."
        )

        # close window
        execute_ui["window"].destroy()

    except Exception as e:
        messagebox.showerror("Complete Error", str(e))

# get all patient history
def get_all_patient_history():
    conn = get_db_connection()
    cursor = conn.cursor()

    # newest first
    cursor.execute("""
        SELECT *
        FROM patient_history
        ORDER BY id DESC
    """)

    data = cursor.fetchall()
    conn.close()
    return data

# delete one history item
def delete_history_item(history_id):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute(
        "DELETE FROM patient_history WHERE id=?",
        (history_id,)
    )

    conn.commit()
    conn.close()

# clear all history
def clear_all_history():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("DELETE FROM patient_history")

    conn.commit()
    conn.close()

# history window button
def handle_view_patient_history(ui, root_window):
    open_all_patient_history_window(root_window)

# in this cell show patient:
duplicate and visit counter and device notes


In [22]:

# check existing patient
def check_existing_patient(name, phone):
    conn=get_db_connection()
    cur=conn.cursor()
    # match by name (case-insensitive) or phone
    cur.execute("""
        SELECT * FROM patients
        WHERE LOWER(name)=LOWER(?) OR phone=?
        LIMIT 1
    """,(name,phone))
    p=cur.fetchone()
    conn.close()
    return p

# get full visit history for a patient
def get_patient_full_visits(patient_id):
    conn=get_db_connection()
    cur=conn.cursor()
    # newest first
    cur.execute("""
        SELECT * FROM patient_history
        WHERE patient_id=?
        ORDER BY id DESC
    """,(patient_id,))
    data=cur.fetchall()
    conn.close()
    return data

# add patient button
def handle_add_patient(ui):
    # get data from form
    name=ui['entry_name'].get().strip()
    phone=ui['entry_phone'].get().strip()
    age_text=ui['entry_age'].get().strip()
    payment_status=ui['payment_status'].get() if 'payment_status' in ui else "Not Paid"

    # check empty name
    if not name:
        messagebox.showwarning("Missing","Please enter patient name")
        return

    # check age is a number
    try:
        age=int(age_text)
    except:
        messagebox.showerror("Age Error","Age must be a number")
        return

    # check duplicate patient
    old=check_existing_patient(name,phone)

    if old:
        # get previous visits
        visits=get_patient_full_visits(old['id'])
        messagebox.showwarning(
            "Existing Patient",
            f"This patient already exists.\n\n"
            f"Name: {old['name']}\n"
            f"Phone: {old['phone']}\n"
            f"Age: {old['age']}\n"
            f"Previous visits: {len(visits)}\n\n"
            f"Please open history instead of creating duplicate."
        )
        return

    # save new patient + clear form
    save_patient_to_db(name,phone,age,payment_status)
    ui['entry_name'].delete(0,tk.END)
    ui['entry_phone'].delete(0,tk.END)
    ui['entry_age'].delete(0,tk.END)
    refresh_patient_queue(ui)
    messagebox.showinfo("Success","Patient added")

# add required_visits column if missing
def add_patient_visit_columns():
    conn=get_db_connection()
    cur=conn.cursor()
    # check existing columns
    cur.execute("PRAGMA table_info(patients)")
    cols=[x[1] for x in cur.fetchall()]
    # add required_visits if missing
    if "required_visits" not in cols:
        cur.execute("ALTER TABLE patients ADD COLUMN required_visits INTEGER DEFAULT 0")
    conn.commit()
    conn.close()

# save recovery step with doctor note
def save_recovery_step_with_note(patient_id, step_order, description, device, duration, note):
    conn=get_db_connection()
    cur=conn.cursor()
    # insert step with note
    cur.execute("""
        INSERT INTO recovery_steps
        (patient_id,step_order,description,device_name,duration_mins,note)
        VALUES(?,?,?,?,?,?)
    """,(patient_id,step_order,description,device,duration,note))
    conn.commit()
    conn.close()

# add note column to recovery_steps if missing
def create_recovery_note_column():
    conn=get_db_connection()
    cur=conn.cursor()
    # check existing columns
    cur.execute("PRAGMA table_info(recovery_steps)")
    cols=[x[1] for x in cur.fetchall()]
    # add note if missing
    if "note" not in cols:
        cur.execute("ALTER TABLE recovery_steps ADD COLUMN note TEXT")
    conn.commit()
    conn.close()

# run column checks on load
add_patient_visit_columns()
create_recovery_note_column()

# this cell defines the main functions that:
read table columns, add any missing ones, create the visit_log and audit_log tables, restore archived patients, and recompute visit counts without resetting newer totals. 
**Note**: for many infomation read the code 

In [23]:
# these functions supersede the experimental handlers above without erasing old notes
# patient identity persists; each visit is recorded separately

# get column names for a table
def _columns(connection, table):
    return {row[1] for row in connection.execute('PRAGMA table_info(' + table + ')')}

# non-destructive migration of earlier clinic.db versions
def migrate_clinic_database():
    """non-destructive migration of earlier clinic.db versions; safe to rerun"""
    setup_database()
    with get_db_connection() as con:
        # add missing columns to existing tables
        for table, columns in {
            'patients': {'visit_count': 'INTEGER DEFAULT 0',
                         'total_prescribed_visits': 'INTEGER DEFAULT 0'},
            'recovery_steps': {'note': "TEXT DEFAULT ''"},
            'patient_history': {'notes': "TEXT DEFAULT ''"},
        }.items():
            existing = _columns(con, table)
            for col, sql_type in columns.items():
                if col not in existing:
                    con.execute(f'ALTER TABLE {table} ADD COLUMN {col} {sql_type}')
        # create visit_log table
        con.execute('''CREATE TABLE IF NOT EXISTS visit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT, patient_id INTEGER,
            visit_date TEXT, status TEXT, notes TEXT, created_by TEXT, created_at TEXT)''')
        # create audit_log table
        con.execute('''CREATE TABLE IF NOT EXISTS audit_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT, action_type TEXT,
            table_name TEXT, record_id INTEGER, old_value TEXT,
            new_value TEXT, performed_by TEXT, performed_at TEXT)''')
        # earlier versions REMOVED patients after adding them to patient_history
        # restore those archived identities with their original IDs so duplicate
        # detection, appointments, recovery plans and visit records stay connected
        archived = con.execute('''SELECT h.* FROM patient_history h
            JOIN (SELECT patient_id, MAX(id) AS last_id FROM patient_history
                  WHERE patient_id IS NOT NULL GROUP BY patient_id) latest
               ON h.id = latest.last_id
            LEFT JOIN patients p ON p.id = h.patient_id
            WHERE p.id IS NULL ORDER BY h.patient_id''').fetchall()
        for row in archived:
            con.execute('''INSERT OR IGNORE INTO patients
                (id, name, phone, age, status, payment_status, date_added,
                 visit_count, total_prescribed_visits)
                VALUES (?, ?, ?, ?, 'Done', ?, ?, 0, 0)''',
                (row['patient_id'], row['name'], row['phone'], row['age'],
                 row['payment_status'], row['visit_date']))
        # never reset newer counts; older history records may predate visit_log
        con.execute('''UPDATE patients SET visit_count = MAX(
            COALESCE(visit_count, 0),
            (SELECT COUNT(*) FROM patient_history WHERE patient_id=patients.id),
            (SELECT COUNT(*) FROM visit_log
             WHERE patient_id=patients.id AND status='Done'))''')

# write an audit entry inside a transaction
def _audit_in_transaction(con, action, table, record_id, old, new, actor):
    con.execute('''INSERT INTO audit_log
        (action_type,table_name,record_id,old_value,new_value,performed_by,performed_at)
        VALUES (?,?,?,?,?,?,?)''',
        (action, table, record_id, str(old), str(new), actor,
         _dt.datetime.now().isoformat(timespec='seconds')))

# get a single patient by id
def get_patient(patient_id):
    with get_db_connection() as con:
        return con.execute('SELECT * FROM patients WHERE id=?', (int(patient_id),)).fetchone()

# normalize a name for matching
def _name_key(value):
    return ' '.join(str(value or '').casefold().split())

# normalize a phone for matching
def _phone_key(value):
    return ''.join(_re.findall(r'\d+', str(value or '')))

# find matching patients by name OR phone (never auto-merge)
def find_patient_matches(name, phone):
    """match by normalized name OR nonempty phone; offer choices, never auto-merge"""
    target_name, target_phone = _name_key(name), _phone_key(phone)
    with get_db_connection() as con:
        patients = con.execute('SELECT * FROM patients ORDER BY name, id').fetchall()
    matches = []
    for p in patients:
        same_name = bool(target_name and _name_key(p['name']) == target_name)
        same_phone = bool(target_phone and _phone_key(p['phone']) == target_phone)
        if same_name or same_phone:
            reason = ('Same name and phone' if same_name and same_phone
                      else 'Same name' if same_name else 'Same phone')
            matches.append((p, reason))
    return sorted(matches, key=lambda x: (0 if x[1] == 'Same name and phone' else 1, x[0]['id']))

# check existing patient (first match)
def check_existing_patient(name, phone):
    matches = find_patient_matches(name, phone)
    return matches[0][0] if matches else None

# save new patient to database
def save_patient_to_db(name, phone, age, payment_status='Unpaid'):
    with get_db_connection() as con:
        cur = con.execute('''INSERT INTO patients
            (name, phone, age, payment_status, status, date_added,
             visit_count, total_prescribed_visits)
            VALUES (?, ?, ?, ?, 'Waiting', ?, 0, 0)''',
            (name.strip(), str(phone).strip(), int(age), payment_status,
             _dt.date.today().isoformat()))
        return cur.lastrowid

# get visit stats for a patient
def get_patient_visit_stats(patient_id):
    with get_db_connection() as con:
        p = con.execute('SELECT * FROM patients WHERE id=?', (int(patient_id),)).fetchone()
        # stop if patient missing
        if p is None:
            raise ValueError('Patient record not found')
        # count from history + visit_log + stored counter
        history = con.execute('SELECT COUNT(*) FROM patient_history WHERE patient_id=?', (patient_id,)).fetchone()[0]
        done_logs = con.execute('''SELECT COUNT(*) FROM visit_log
             WHERE patient_id=? AND status='Done' ''', (patient_id,)).fetchone()[0]
        attended = max(history, done_logs, int(p['visit_count'] or 0))
        prescribed = max(0, int(p['total_prescribed_visits'] or 0))
        return {'prescribed': prescribed, 'attended': attended,
                'remaining': max(0, prescribed - attended)}

# get completed visit count
def get_completed_visit_count(patient_id):
    return get_patient_visit_stats(patient_id)['attended']

# get total prescribed visits
def get_total_prescribed_visits(patient_id):
    return get_patient_visit_stats(patient_id)['prescribed']

# get remaining visits
def get_remaining_visits(patient_id):
    return get_patient_visit_stats(patient_id)['remaining']

# get visit count (compat wrapper)
def get_patient_visit_count(patient_id):
    return get_completed_visit_count(patient_id)

# get clinic-wide overview totals
def get_clinic_overview():
    with get_db_connection() as con:
        ids = [r[0] for r in con.execute('SELECT id FROM patients')]
    stats = [get_patient_visit_stats(pid) for pid in ids]
    return {'patients': len(ids),
            'prescribed': sum(s['prescribed'] for s in stats),
            'attended': sum(s['attended'] for s in stats),
            'remaining': sum(s['remaining'] for s in stats)}

# refresh queue table
def refresh_patient_queue(ui):
    tree = ui['patient_tree']
    # remove old rows
    for row in tree.get_children():
        tree.delete(row)
    with get_db_connection() as con:
        patients = con.execute('''SELECT * FROM patients
            WHERE status='Waiting' ORDER BY id''').fetchall()
    # insert each waiting patient
    for p in patients:
        stats = get_patient_visit_stats(p['id'])
        tree.insert('', tk.END, iid=str(p['id']), values=(
            p['id'], p['name'], p['age'], p['phone'], p['payment_status'],
            stats['prescribed'], stats['attended'], stats['remaining']))
    # update overview label
    if 'overview_label' in ui:
        s = get_clinic_overview()
        ui['overview_label'].config(text=(
            f"Patient records: {s['patients']}    |    Doctor prescribed: {s['prescribed']}    "
            f"|    Visits attended: {s['attended']}    |    Visits remaining: {s['remaining']}"))

# select + scroll to a patient in queue
def _focus_patient_in_queue(ui, patient_id):
    tree = ui['patient_tree']
    iid = str(patient_id)
    if tree.exists(iid):
        tree.selection_set(iid)
        tree.focus(iid)
        tree.see(iid)

# requeue a patient (status → waiting)
def requeue_patient(patient_id):
    with get_db_connection() as con:
        patient = con.execute('SELECT status FROM patients WHERE id=?', (patient_id,)).fetchone()
        # stop if patient missing
        if patient is None:
            raise ValueError('Patient record not found')
        if patient['status'] != 'Waiting':
            con.execute("UPDATE patients SET status='Waiting' WHERE id=?", (patient_id,))
            _audit_in_transaction(con, 'REQUEUE', 'patients', patient_id,
                                  patient['status'], 'Waiting', 'Nurse')

# clear nurse form fields
def _clear_nurse_form(ui):
    for field in ('entry_name', 'entry_phone', 'entry_age'):
        ui[field].delete(0, tk.END)

# update patient record
def update_patient_record(patient_id, name, phone, age, payment_status,
                          prescribed_visits=None, actor='Staff'):
    name = str(name).strip()
    phone = str(phone or '').strip()
    # validate name
    if not name:
        raise ValueError('Patient name cannot be empty.')
    # validate age
    try:
        age = int(age)
    except (TypeError, ValueError):
        raise ValueError('Age must be a whole number.')
    if not 0 <= age <= 120:
        raise ValueError('Age must be between 0 and 120.')
    # validate payment
    if payment_status not in ('Unpaid', 'Paid', 'Partial', 'Not Paid'):
        raise ValueError('Choose a valid payment status.')
    # validate prescribed visits
    if prescribed_visits is not None:
        try:
            prescribed_visits = int(prescribed_visits)
        except (ValueError, TypeError):
            raise ValueError('Visits prescribed must be a nonnegative whole number.')
        if prescribed_visits < 0:
            raise ValueError('Visits prescribed cannot be negative.')
    with get_db_connection() as con:
        old = con.execute('SELECT * FROM patients WHERE id=?', (patient_id,)).fetchone()
        # stop if patient missing
        if old is None:
            raise ValueError('Patient record not found.')
        total = (int(old['total_prescribed_visits'] or 0)
                 if prescribed_visits is None else prescribed_visits)
        # update patients table
        con.execute('''UPDATE patients SET name=?, phone=?, age=?,
            payment_status=?, total_prescribed_visits=? WHERE id=?''',
            (name, phone, age, payment_status, total, patient_id))
        # keep displayed demographics in older visitor-history entries in sync
        con.execute('''UPDATE patient_history SET name=?, phone=?, age=?,
            payment_status=? WHERE patient_id=?''',
            (name, phone, age, payment_status, patient_id))
        # log the edit
        _audit_in_transaction(con, 'EDIT', 'patients', patient_id,
            dict(old), {'name': name, 'phone': phone, 'age': age,
                        'payment_status': payment_status,
                        'total_prescribed_visits': total}, actor)

# open patient editor dialog
def open_patient_editor(parent, patient_id, after_save=None):
    patient = get_patient(patient_id)
    # stop if patient missing
    if patient is None:
        messagebox.showerror('Not Found', 'The patient is no longer in the database.', parent=parent)
        return
    s = get_patient_visit_stats(patient_id)
    widgets = build_patient_edit_dialog(parent, patient, s)
    # save handler
    def save():
        try:
            update_patient_record(
                patient_id, widgets['entry_name'].get(),
                widgets['entry_phone'].get(), widgets['entry_age'].get(),
                widgets['combo_payment'].get(), widgets['entry_total_visits'].get())
        except ValueError as exc:
            messagebox.showerror('Invalid Information', str(exc), parent=widgets['window'])
            return
        widgets['window'].destroy()
        if after_save:
            after_save()
    widgets['btn_save'].config(command=save)

# open duplicate patient warning
def open_duplicate_patient_warning(ui, matches):
    """clickable records, including former visitors, with update and queue actions"""
    parent = ui['left_frame'].winfo_toplevel()
    dialog = build_duplicate_patient_dialog(parent)
    tree = dialog['tree_matches']
    # populate matches table
    def populate():
        for item in tree.get_children():
            tree.delete(item)
        for patient, reason in matches:
            live = get_patient(patient['id'])
            if not live:
                continue
            s = get_patient_visit_stats(live['id'])
            tree.insert('', tk.END, iid=str(live['id']), values=(
                live['id'], live['name'], live['phone'] or '', live['age'],
                live['payment_status'], live['status'],
                s['prescribed'], s['attended'], s['remaining'], reason))
        if tree.get_children():
            tree.selection_set(tree.get_children()[0])
    # get chosen patient id
    def chosen():
        selected = tree.selection()
        if not selected:
            messagebox.showwarning('Choose Patient', 'Select a patient name first.', parent=dialog['window'])
            return None
        return int(tree.item(selected[0], 'values')[0])
    # use selected patient (requeue)
    def use_patient():
        pid = chosen()
        if pid is None:
            return
        requeue_patient(pid)
        refresh_patient_queue(ui)
        _focus_patient_in_queue(ui, pid)
        _clear_nurse_form(ui)
        dialog['window'].destroy()
    # edit selected patient
    def edit_patient():
        pid = chosen()
        if pid is not None:
            open_patient_editor(dialog['window'], pid,
                after_save=lambda: (populate(), refresh_patient_queue(ui)))
    # create new (different person)
    def different_patient():
        if not messagebox.askyesno('Confirm Different Person',
                'You are creating a NEW patient even though similar records exist.\n'
                'Only proceed if this is a different person.', parent=dialog['window']):
            return
        try:
            name = ui['entry_name'].get().strip()
            age = int(ui['entry_age'].get())
            new_id = save_patient_to_db(name, ui['entry_phone'].get().strip(),
                                        age, ui['combo_payment'].get())
            refresh_patient_queue(ui)
            _focus_patient_in_queue(ui, new_id)
            _clear_nurse_form(ui)
            dialog['window'].destroy()
        except Exception as exc:
            messagebox.showerror('Save Failed', str(exc), parent=dialog['window'])
    # connect buttons
    dialog['btn_use'].config(command=use_patient)
    dialog['btn_edit'].config(command=edit_patient)
    dialog['btn_new'].config(command=different_patient)
    tree.bind('<Double-1>', lambda event: use_patient())
    populate()
    dialog['window'].grab_set()

# add patient button
def handle_add_patient(ui):
    # get data from form
    name = ui['entry_name'].get().strip()
    phone = ui['entry_phone'].get().strip()
    # check name
    if not name:
        messagebox.showwarning('Missing Information', 'Please enter the patient name.')
        return
    # check age
    try:
        age = int(ui['entry_age'].get().strip())
    except ValueError:
        messagebox.showerror('Invalid Age', 'Age must be a whole number.')
        return
    if not 0 <= age <= 120:
        messagebox.showerror('Invalid Age', 'Age must be between 0 and 120.')
        return
    # check duplicates first
    matches = find_patient_matches(name, phone)
    if matches:
        open_duplicate_patient_warning(ui, matches)
        return
    try:
        payment = ui['combo_payment'].get()
        patient_id = save_patient_to_db(name, phone, age, payment)
        refresh_patient_queue(ui)
        _focus_patient_in_queue(ui, patient_id)
        _clear_nurse_form(ui)
    except Exception as exc:
        messagebox.showerror('Database Error', str(exc))

# remove from queue without deleting record
def handle_delete_patient(ui):
    """remove from queue without deleting the permanent patient record/history"""
    pid = _get_selected_queue_patient(ui)
    if pid is None:
        return
    patient = get_patient(pid)
    # confirm removal
    if not messagebox.askyesno('Remove from Waiting Queue',
        f"Remove {patient['name']} from the waiting queue?\n"
        'Patient details, attendance history and doctor notes will be retained.'):
        return
    with get_db_connection() as con:
        con.execute("UPDATE patients SET status='Removed' WHERE id=?", (pid,))
        _audit_in_transaction(con, 'REMOVE_QUEUE', 'patients', pid,
                              'Waiting', 'Removed', 'Nurse')
    refresh_patient_queue(ui)

# get selected queue patient id
def _get_selected_queue_patient(ui):
    selected = ui['patient_tree'].selection()
    # warn + stop if no selection
    if not selected:
        messagebox.showwarning('Select Patient', 'Select a patient in the waiting queue first.')
        return None
    return int(ui['patient_tree'].item(selected[0], 'values')[0])

# create recovery plan button
def handle_create_plan(ui, root_window):
    pid = _get_selected_queue_patient(ui)
    if pid is not None:
        patient = get_patient(pid)
        open_recovery_plan_window(root_window, pid, patient['name'])

# open recovery plan window + connect buttons
def open_recovery_plan_window(parent, patient_id, patient_name):
    widgets = build_recovery_plan_window(parent, patient_name)
    tree = widgets['tree_steps']
    stats = get_patient_visit_stats(patient_id)
    widgets['entry_total_visits'].insert(0, str(stats['prescribed'] or max(1, stats['attended'])))
    # load existing steps
    existing = get_recovery_steps_for_patient(patient_id)
    for i, step in enumerate(existing, start=1):
        tree.insert('', tk.END, values=(
            i, step['description'], step['device_name'], step['duration_mins'], step['note'] or ''))
    # clear editor fields
    def reset_fields():
        for k in ('entry_desc', 'entry_duration'):
            widgets[k].delete(0, tk.END)
        widgets['combo_device'].set('')
        widgets['entry_step_note'].delete('1.0', tk.END)
        tree.selection_remove(tree.selection())
    # load selected step into editor
    def edit_selected(event=None):
        selected = tree.selection()
        if not selected:
            return
        values = tree.item(selected[0], 'values')
        widgets['entry_desc'].delete(0, tk.END)
        widgets['entry_duration'].delete(0, tk.END)
        widgets['entry_step_note'].delete('1.0', tk.END)
        widgets['entry_desc'].insert(0, values[1])
        widgets['combo_device'].set(values[2])
        widgets['entry_duration'].insert(0, str(values[3]))
        widgets['entry_step_note'].insert('1.0', values[4])
    # add or update a step
    def add_or_update():
        desc = widgets['entry_desc'].get().strip()
        device = widgets['combo_device'].get().strip()
        note = widgets['entry_step_note'].get('1.0', 'end-1c').strip()
        # validate duration
        try:
            duration = int(widgets['entry_duration'].get().strip())
            if duration <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror('Invalid Duration', 'Enter a positive number of minutes.', parent=widgets['window'])
            return
        # validate step
        if not desc or not device:
            messagebox.showwarning('Missing Step', 'Enter a step description and select a device.', parent=widgets['window'])
            return
        selected = tree.selection()
        # update existing or add new
        if selected:
            item = selected[0]
            order = tree.item(item, 'values')[0]
            tree.item(item, values=(order, desc, device, duration, note))
        else:
            tree.insert('', tk.END, values=(len(tree.get_children()) + 1,
                                            desc, device, duration, note))
        reset_fields()
    # remove selected steps
    def remove_selected():
        for item in tree.selection():
            tree.delete(item)
        # renumber remaining steps
        for i, item in enumerate(tree.get_children(), 1):
            values = tree.item(item, 'values')
            tree.item(item, values=(i, *values[1:]))
        reset_fields()
    # save plan to database
    def save_plan():
        items = tree.get_children()
        # warn if empty
        if not items:
            messagebox.showwarning('Empty Plan', 'Add at least one treatment step.', parent=widgets['window'])
            return
        # validate prescribed visits
        try:
            total = int(widgets['entry_total_visits'].get())
            if total < 1:
                raise ValueError
        except ValueError:
            messagebox.showerror('Invalid Visit Count', 'Doctor must prescribe at least one visit.', parent=widgets['window'])
            return
        try:
            with get_db_connection() as con:
                con.execute('UPDATE patients SET total_prescribed_visits=? WHERE id=?', (total, patient_id))
                # replacing instead of appending avoids duplicate steps when re-editing
                con.execute('DELETE FROM recovery_steps WHERE patient_id=?', (patient_id,))
                for position, item in enumerate(items, 1):
                    _, desc, device, duration, note = tree.item(item, 'values')
                    con.execute('''INSERT INTO recovery_steps
                        (patient_id,step_order,description,device_name,duration_mins,note)
                        VALUES (?,?,?,?,?,?)''',
                        (patient_id, position, desc, device, int(duration), note))
                _audit_in_transaction(con, 'SAVE_PLAN', 'recovery_steps',
                    patient_id, f'{len(existing)} steps',
                    f'{len(items)} steps; prescribed visits: {total}', 'Doctor')
        except Exception as exc:
            messagebox.showerror('Save Failed', str(exc), parent=widgets['window'])
            return
        widgets['window'].destroy()
        messagebox.showinfo('Plan Saved', f'Treatment plan saved. Prescribed visits: {total}.')
    # connect buttons
    widgets['btn_add_step'].config(command=add_or_update)
    widgets['btn_remove_step'].config(command=remove_selected)
    widgets['btn_save_plan'].config(command=save_plan)
    tree.bind('<<TreeviewSelect>>', edit_selected)

# view execution plan button
def handle_view_execution_plan(ui, root_window):
    pid = _get_selected_queue_patient(ui)
    if pid is not None:
        open_execute_plan_window(root_window, pid, get_patient(pid)['name'], ui)

# open execute plan window + connect buttons
def open_execute_plan_window(parent, patient_id, patient_name, main_ui):
    steps = get_recovery_steps_for_patient(patient_id)
    # warn if no plan
    if not steps:
        messagebox.showwarning('No Recovery Plan', 'The doctor needs to create a recovery plan first.')
        return
    widgets = build_execute_plan_window(parent, patient_name, patient_id)
    tree = widgets['tree_steps']
    s = get_patient_visit_stats(patient_id)
    # show visit stats
    widgets['visit_label'].config(text=(
        f"Prescribed: {s['prescribed']}  |  Attended: {s['attended']}  |  Remaining: {s['remaining']}"))
    # load steps with pending status
    for step in steps:
        tree.insert('', tk.END, values=(step['step_order'], step['description'],
            step['device_name'], step['duration_mins'], step['note'] or '', 'Pending'))
    # show note for selected step
    def update_step_note(event=None):
        sel = tree.selection()
        if sel:
            values = tree.item(sel[0], 'values')
            widgets['instruction_label'].config(text=f'Step {values[0]}: {values[4] or "No additional doctor notes."}')
    # mark next pending step done
    def mark_next():
        all_items = tree.get_children()
        pending = [item for item in all_items if tree.item(item, 'values')[-1] != 'Done']
        # all done
        if not pending:
            messagebox.showinfo('Steps Complete', 'All treatment steps have been marked done.', parent=widgets['window'])
            return
        selected = tree.selection()
        # enforce step order
        if not selected or selected[0] != pending[0]:
            messagebox.showwarning('Follow Step Order',
                f'Select step {tree.item(pending[0], "values")[0]} and complete steps in order.',
                parent=widgets['window'])
            tree.selection_set(pending[0])
            return
        values = tree.item(pending[0], 'values')
        tree.item(pending[0], values=(*values[:-1], 'Done'))
        tree.selection_remove(tree.selection())
        # select next pending
        remaining = [item for item in all_items if tree.item(item, 'values')[-1] != 'Done']
        if remaining:
            tree.selection_set(remaining[0])
            tree.see(remaining[0])
    # complete visit
    def complete_visit():
        # warn if steps unfinished
        if any(tree.item(item, 'values')[-1] != 'Done' for item in tree.get_children()):
            messagebox.showwarning('Unfinished Treatment',
                'Complete all steps in order before finishing this visit.', parent=widgets['window'])
            return
        try:
            stats = record_completed_visit(patient_id, widgets['entry_visit_note'].get('1.0', 'end-1c').strip())
        except Exception as exc:
            messagebox.showerror('Visit Not Saved', str(exc), parent=widgets['window'])
            return
        refresh_patient_queue(main_ui)
        widgets['window'].destroy()
        messagebox.showinfo('Visit Recorded',
            f"Attended visits: {stats['attended']}\nRemaining visits: {stats['remaining']}")
    # connect handlers
    tree.bind('<<TreeviewSelect>>', update_step_note)
    widgets['btn_mark_step'].config(command=mark_next)
    widgets['btn_complete'].config(command=complete_visit)

# record a completed visit (single atomic transaction)
def record_completed_visit(patient_id, notes=''):
    """single atomic transaction for history, visit log, counters and queue status"""
    timestamp = _dt.datetime.now().isoformat(sep=' ', timespec='seconds')
    with get_db_connection() as con:
        p = con.execute('SELECT * FROM patients WHERE id=?', (patient_id,)).fetchone()
        # stop if not currently waiting
        if p is None or p['status'] != 'Waiting':
            raise ValueError('Patient is not currently waiting. Queue the patient before completing a visit.')
        # save to history
        con.execute('''INSERT INTO patient_history
            (patient_id,name,phone,age,payment_status,visit_date,notes)
            VALUES (?,?,?,?,?,?,?)''',
            (patient_id, p['name'], p['phone'], p['age'], p['payment_status'], timestamp, notes))
        # save to visit log
        con.execute('''INSERT INTO visit_log
            (patient_id,visit_date,status,notes,created_by,created_at)
            VALUES (?,?,'Done',?,'Nurse',?)''', (patient_id, timestamp, notes, timestamp))
        # recompute visit count from all sources
        h_count = con.execute('SELECT COUNT(*) FROM patient_history WHERE patient_id=?', (patient_id,)).fetchone()[0]
        l_count = con.execute("SELECT COUNT(*) FROM visit_log WHERE patient_id=? AND status='Done'", (patient_id,)).fetchone()[0]
        new_count = max(h_count, l_count, int(p['visit_count'] or 0) + 1)
        con.execute('''UPDATE patients SET status='Done', visit_count=? WHERE id=?''', (new_count, patient_id))
        _audit_in_transaction(con, 'COMPLETE_VISIT', 'patients', patient_id,
                              p['status'], f'Done; attended={new_count}', 'Nurse')
    return get_patient_visit_stats(patient_id)

# complete patient button (compat wrapper)
def handle_complete_patient(execute_ui, main_ui):
    # support older callers while enforcing completion in the current ui
    tree = execute_ui['tree_steps']
    if not tree.get_children() or any(tree.item(item, 'values')[-1] != 'Done'
                                      for item in tree.get_children()):
        messagebox.showwarning('Incomplete', 'Finish all steps before completing the visit.')
        return
    try:
        record_completed_visit(execute_ui['patient_id'])
        refresh_patient_queue(main_ui)
        execute_ui['window'].destroy()
    except Exception as exc:
        messagebox.showerror('Visit Error', str(exc))

# refresh the patient history window
def _refresh_patient_history_window(widgets):
    patient_tree = widgets['tree_patients']
    old_selected = patient_tree.selection()
    old_id = old_selected[0] if old_selected else None
    # remove old rows
    for item in patient_tree.get_children():
        patient_tree.delete(item)
    with get_db_connection() as con:
        patients = con.execute('SELECT * FROM patients ORDER BY name, id').fetchall()
    # insert all patients with stats
    for p in patients:
        s = get_patient_visit_stats(p['id'])
        patient_tree.insert('', tk.END, iid=str(p['id']), values=(
            p['id'], p['name'], p['phone'], p['age'], p['status'],
            s['prescribed'], s['attended'], s['remaining']))
    # restore selection
    if old_id and patient_tree.exists(old_id):
        patient_tree.selection_set(old_id)
    elif patient_tree.get_children():
        patient_tree.selection_set(patient_tree.get_children()[0])
    _show_history_selection(widgets)

# show selected patient details in history window
def _show_history_selection(widgets):
    sel = widgets['tree_patients'].selection()
    # clear visits + steps tables
    for key in ('tree_visits', 'tree_steps'):
        tree = widgets[key]
        for item in tree.get_children():
            tree.delete(item)
    # nothing selected
    if not sel:
        widgets['patient_detail_label'].config(text='Choose a patient to see visits and treatment notes.')
        return
    pid = int(sel[0])
    p = get_patient(pid)
    s = get_patient_visit_stats(pid)
    # detail label
    widgets['patient_detail_label'].config(text=(
        f"{p['name']}  |  Phone: {p['phone'] or '-'}  |  Age: {p['age']}  |  "
        f"Payment: {p['payment_status']}\nDoctor prescribed: {s['prescribed']}   "
        f"Visits attended: {s['attended']}   Visits remaining: {s['remaining']}"))
    # load visits
    with get_db_connection() as con:
        visits = con.execute('''SELECT * FROM patient_history
            WHERE patient_id=? ORDER BY visit_date DESC, id DESC''', (pid,)).fetchall()
    for v in visits:
        widgets['tree_visits'].insert('', tk.END,
            values=(v['id'], v['visit_date'], v['payment_status'], v['notes'] or ''))
    # load steps
    for step in get_recovery_steps_for_patient(pid):
        widgets['tree_steps'].insert('', tk.END, values=(
            step['step_order'], step['description'], step['device_name'],
            step['duration_mins'], step['note'] or ''))

# history window button
def handle_view_patient_history(ui, root_window):
    widgets = build_patient_history_window(root_window)
    # get selected patient id
    def selected_pid():
        selected = widgets['tree_patients'].selection()
        if not selected:
            messagebox.showwarning('No Selection', 'Select a patient first.', parent=widgets['window'])
            return None
        return int(selected[0])
    # reload history + queue
    def reload():
        _refresh_patient_history_window(widgets)
        refresh_patient_queue(ui)
    # edit selected patient
    def edit():
        pid = selected_pid()
        if pid is not None:
            open_patient_editor(widgets['window'], pid, after_save=reload)
    # add selected patient back to queue
    def add_to_queue():
        pid = selected_pid()
        if pid is not None:
            requeue_patient(pid)
            reload()
            _focus_patient_in_queue(ui, pid)
            messagebox.showinfo('Queue Updated', 'Existing patient selected in the waiting queue.', parent=widgets['window'])
    # connect handlers
    widgets['tree_patients'].bind('<<TreeviewSelect>>',
        lambda event: _show_history_selection(widgets))
    widgets['btn_edit'].config(command=edit)
    widgets['btn_queue'].config(command=add_to_queue)
    widgets['btn_close'].config(command=widgets['window'].destroy)
    reload()

# build patient report rows for excel
def _patient_report_rows():
    with get_db_connection() as con:
        patients = con.execute('SELECT * FROM patients ORDER BY id').fetchall()
    result = []
    for p in patients:
        s = get_patient_visit_stats(p['id'])
        result.append({'Patient ID': p['id'], 'Name': p['name'], 'Phone': p['phone'] or '',
            'Age': p['age'], 'Payment': p['payment_status'], 'Status': p['status'],
            'Date Added': p['date_added'], 'Visits Prescribed': s['prescribed'],
            'Visits Completed': s['attended'], 'Visits Remaining': s['remaining']})
    return result

# export clinic report to excel
def export_clinic_report(filename):
    """round-trip-compatible patients sheet plus read-only history/step sheets"""
    with get_db_connection() as con:
        # visit history rows
        visit_rows = [dict(row) for row in con.execute('''SELECT
            id AS 'History ID', patient_id AS 'Patient ID', name AS Name,
            phone AS Phone, age AS Age, payment_status AS Payment,
            visit_date AS 'Visit Date', notes AS 'Visit Notes'
            FROM patient_history ORDER BY id DESC''')]
        # recovery step rows
        step_rows = [dict(row) for row in con.execute('''SELECT patient_id AS 'Patient ID',
            step_order AS 'Step', description AS 'Description',
            device_name AS 'Device', duration_mins AS 'Duration (minutes)',
            note AS 'Doctor Step Notes' FROM recovery_steps
            ORDER BY patient_id,step_order''')]
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # patients sheet
        pd.DataFrame(_patient_report_rows(), columns=[
            'Patient ID', 'Name', 'Phone', 'Age', 'Payment', 'Status',
            'Date Added', 'Visits Prescribed', 'Visits Completed', 'Visits Remaining',
        ]).to_excel(writer, index=False, sheet_name='Patients')
        # visit history sheet
        pd.DataFrame(visit_rows, columns=[
            'History ID', 'Patient ID', 'Name', 'Phone', 'Age', 'Payment',
            'Visit Date', 'Visit Notes']).to_excel(writer, index=False, sheet_name='Visit History')
        # recovery steps sheet
        pd.DataFrame(step_rows, columns=[
            'Patient ID', 'Step', 'Description', 'Device',
            'Duration (minutes)', 'Doctor Step Notes']).to_excel(
            writer, index=False, sheet_name='Recovery Steps')
        # freeze panes + autofilter + auto width
        for sheet in writer.sheets.values():
            sheet.freeze_panes = 'A2'
            sheet.auto_filter.ref = sheet.dimensions
            for column in sheet.columns:
                letter = column[0].column_letter
                sheet.column_dimensions[letter].width = min(44, max(14,
                    max(len(str(c.value or '')) for c in column) + 2))
        # keep leading zeros in phone numbers as text
        for sheet_name in ('Patients', 'Visit History'):
            ws = writer.sheets[sheet_name]
            phone_col = next((c.column for c in ws[1] if c.value == 'Phone'), None)
            if phone_col:
                for col_cells in ws.iter_cols(min_col=phone_col, max_col=phone_col, min_row=2):
                    for cell in col_cells:
                        if cell.value is not None:
                            cell.value = str(cell.value)
                            cell.number_format = '@'

# export to excel button
def handle_export_to_excel():
    filename = filedialog.asksaveasfilename(defaultextension='.xlsx',
        filetypes=[('Excel Workbook', '*.xlsx')], title='Save Clinic Report')
    if filename:
        try:
            export_clinic_report(filename)
            messagebox.showinfo('Excel Saved', f'Clinic report exported to:\n{filename}')
        except Exception as exc:
            messagebox.showerror('Export Failed', str(exc))

# import patient updates from excel
def import_patient_updates_from_excel(filename):
    """update existing ids only; never create duplicates or overwrite visit logs"""
    workbook = pd.ExcelFile(filename)
    sheet = ('Patients' if 'Patients' in workbook.sheet_names else
             'Patient Data' if 'Patient Data' in workbook.sheet_names else None)
    # stop if no patients sheet
    if not sheet:
        raise ValueError('Excel file must contain a Patients sheet.')
    df = pd.read_excel(workbook, sheet_name=sheet, dtype=str)
    # rename old column names
    aliases = {'id':'Patient ID', 'name':'Name', 'phone':'Phone',
               'age':'Age', 'payment_status':'Payment',
               'total_prescribed_visits':'Visits Prescribed',
               'required_visits':'Visits Prescribed'}
    df = df.rename(columns=aliases)
    # check required columns
    mandatory = {'Patient ID', 'Name', 'Age'}
    if not mandatory.issubset(df.columns):
        raise ValueError('Patients sheet needs Patient ID, Name and Age columns.')
    candidates = []
    seen = set()
    for row_number, row in df.iterrows():
        rid = row.get('Patient ID')
        if pd.isna(rid) or str(rid).strip() == '':
            continue
        # validate patient id
        try:
            pid = int(float(rid))
            if str(rid).strip() not in (str(pid), f'{pid}.0'):
                raise ValueError
        except (ValueError, TypeError):
            raise ValueError(f'Row {row_number + 2}: invalid Patient ID.')
        if pid in seen:
            raise ValueError(f'Patient ID {pid} appears twice in Excel.')
        seen.add(pid)
        p = get_patient(pid)
        # stop if patient missing
        if p is None:
            raise ValueError(f'Patient ID {pid} does not exist. Import cannot create patients.')
        # keep old value if cell empty
        def field(key, old):
            val = row.get(key)
            return old if val is None or pd.isna(val) else str(val).strip()
        name = field('Name', p['name'])
        phone = field('Phone', p['phone'])
        age = field('Age', p['age'])
        payment = field('Payment', p['payment_status'])
        target = field('Visits Prescribed', p['total_prescribed_visits'])
        # validate values
        try:
            age_int, target_int = int(age), int(target)
            if not name or not 0 <= age_int <= 120 or target_int < 0:
                raise ValueError
        except ValueError:
            raise ValueError(f'Row {row_number + 2}: invalid name, age or prescribed visits.')
        if payment not in ('Unpaid', 'Paid', 'Partial', 'Not Paid'):
            raise ValueError(f'Row {row_number + 2}: invalid payment status.')
        candidate = (pid, name, phone, age_int, payment, target_int)
        # only keep changed rows
        if (name, phone, age_int, payment, target_int) != (
                p['name'], p['phone'] or '', p['age'], p['payment_status'],
                int(p['total_prescribed_visits'] or 0)):
            candidates.append(candidate)
    # all changes happen in one transaction after full validation
    with get_db_connection() as con:
        for pid, name, phone, age, payment, target in candidates:
            old = con.execute('SELECT * FROM patients WHERE id=?', (pid,)).fetchone()
            con.execute('''UPDATE patients SET name=?, phone=?, age=?,
                payment_status=?, total_prescribed_visits=? WHERE id=?''',
                (name, phone, age, payment, target, pid))
            # sync demographics in history
            con.execute('''UPDATE patient_history SET name=?, phone=?, age=?,
                payment_status=? WHERE patient_id=?''',
                (name, phone, age, payment, pid))
            _audit_in_transaction(con, 'EXCEL_EDIT', 'patients', pid,
                dict(old), {'name': name, 'phone': phone, 'age': age,
                            'payment_status': payment, 'prescribed_visits': target}, 'Doctor')
    return len(candidates)

# import edited excel button
def handle_import_from_excel(ui):
    filename = filedialog.askopenfilename(filetypes=[('Excel Workbook', '*.xlsx')],
                                          title='Import Updates to Existing Patients')
    if not filename:
        return
    # confirm import
    if not messagebox.askyesno('Review Import',
        'Import edits from the Patients sheet?\n\n'
        'Patients are matched ONLY by Patient ID. Existing names, age, phone, payment and prescribed visits may change.\n'
        'Visit history and clinical step records are not overwritten.'):
        return
    try:
        changed = import_patient_updates_from_excel(filename)
        refresh_patient_queue(ui)
        messagebox.showinfo('Import Complete', f'{changed} patient record(s) updated. No duplicates created.')
    except Exception as exc:
        messagebox.showerror('Import Failed', str(exc))

# run migration on load
migrate_clinic_database()

# connect one-click doctor device choices to ordered nurse recovery steps.
all patient matching, database migration, visit counting and excel workflows come from v3

In [24]:
# open recovery plan window and connect buttons
def open_recovery_plan_window(parent, patient_id, patient_name):
    widgets = build_recovery_plan_window(parent, patient_name)
    tree = widgets['tree_steps']
    stats = get_patient_visit_stats(patient_id)
    widgets['entry_total_visits'].insert(0, str(stats['prescribed'] or max(1, stats['attended'])))
    # load previous steps
    previous = get_recovery_steps_for_patient(patient_id)
    for i, step in enumerate(previous, 1):
        tree.insert('', tk.END, values=(i, step['description'], step['device_name'],
                    step['duration_mins'], step['note'] or ''))

    # update status label
    def status(msg):
        widgets['status_label'].config(text=msg)

    # renumber all steps
    def renumber():
        for pos, item in enumerate(tree.get_children(), 1):
            vals = tree.item(item, 'values')
            tree.item(item, values=(pos, *vals[1:]))

    # clear editor fields
    def reset_editor():
        tree.selection_remove(tree.selection())
        widgets['entry_desc'].delete(0, tk.END)
        widgets['combo_device'].set('')
        widgets['entry_duration'].delete(0, tk.END)
        widgets['entry_step_note'].delete('1.0', tk.END)

    # load selected step into editor
    def selected_editor(event=None):
        selected = tree.selection()
        if not selected:
            return
        values = tree.item(selected[0], 'values')
        widgets['entry_desc'].delete(0, tk.END)
        widgets['entry_desc'].insert(0, str(values[1]))
        widgets['combo_device'].set(str(values[2]))
        widgets['entry_duration'].delete(0, tk.END)
        # restore duration only if valid
        if str(values[3]).isdigit() and int(values[3]) > 0:
            widgets['entry_duration'].insert(0, str(values[3]))
        widgets['entry_step_note'].delete('1.0', tk.END)
        widgets['entry_step_note'].insert('1.0', str(values[4]))
        status(f'Step {values[0]} selected. Edit the time and notes, then click Update Selected Step.')

    # one-click add device as new step
    def quick_add(device):
        # one click immediately creates a step; duration intentionally not guessed
        item = tree.insert('', tk.END, values=(len(tree.get_children()) + 1,
            device, device, 'Set minutes', ''))
        tree.selection_set(item)
        tree.focus(item)
        tree.see(item)
        selected_editor()
        widgets['entry_duration'].focus_set()
        status(f'Added {device}. Set minutes and doctor instructions; click Update Selected Step.')

    # apply editor to selected or new step
    def apply_editor():
        description = widgets['entry_desc'].get().strip()
        device = widgets['combo_device'].get().strip()
        note = widgets['entry_step_note'].get('1.0', 'end-1c').strip()
        # validate duration
        try:
            minutes = int(widgets['entry_duration'].get().strip())
            if minutes < 1:
                raise ValueError
        except (TypeError, ValueError):
            messagebox.showwarning('Set Treatment Time',
                'Enter a positive number of treatment minutes for this step.', parent=widgets['window'])
            return False
        # validate step
        if not description or not device:
            messagebox.showwarning('Incomplete Treatment Step',
                'Enter a treatment description and choose the device.', parent=widgets['window'])
            return False
        selected = tree.selection()
        # update existing or add new
        if selected:
            item = selected[0]
            number = tree.item(item, 'values')[0]
            tree.item(item, values=(number, description, device, minutes, note))
        else:
            tree.insert('', tk.END, values=(len(tree.get_children()) + 1,
                                            description, device, minutes, note))
        renumber()
        reset_editor()
        status(f'Step saved in the list. Add another device or press Save Recovery Plan.')
        return True

    # remove selected steps + renumber
    def remove_selected():
        for item in tree.selection():
            tree.delete(item)
        renumber()
        reset_editor()
        status('Step removed; remaining steps were renumbered.')

    # move selected step up or down
    def move_selected(delta):
        selected = tree.selection()
        # warn if no selection
        if not selected:
            messagebox.showinfo('Choose a Step',
                'Select a step in the ordered list first.', parent=widgets['window'])
            return
        item = selected[0]
        position = tree.index(item)
        new_position = position + delta
        # stop if out of bounds
        if new_position < 0 or new_position >= len(tree.get_children()):
            return
        tree.move(item, '', new_position)
        renumber()
        tree.selection_set(item)
        tree.focus(item)
        tree.see(item)
        status(f'Moved this device to step {new_position + 1}. Nurse will follow this order.')

    # save plan to database
    def save_plan():
        items = tree.get_children()
        # warn if empty
        if not items:
            messagebox.showwarning('No Treatment Steps',
                'Choose at least one device or add a custom treatment step.', parent=widgets['window'])
            return
        # if doctor is still editing a selected row, include latest notes when saving
        selected = tree.selection()
        if selected:
            values = tree.item(selected[0], 'values')
            updated = (widgets['entry_desc'].get().strip(),
                       widgets['combo_device'].get().strip(),
                       widgets['entry_duration'].get().strip(),
                       widgets['entry_step_note'].get('1.0', 'end-1c').strip())
            if updated != tuple(str(v) for v in values[1:]):
                if not apply_editor():
                    return
        # validate prescribed visits
        try:
            visits = int(widgets['entry_total_visits'].get().strip())
            if visits < 1:
                raise ValueError
        except (TypeError, ValueError):
            messagebox.showerror('Invalid Prescribed Visits',
                'The doctor must enter at least one prescribed visit.', parent=widgets['window'])
            return
        valid_steps = []
        for number, item in enumerate(tree.get_children(), 1):
            _, desc, device, duration, note = tree.item(item, 'values')
            # validate duration
            try:
                minutes = int(duration)
                if minutes < 1:
                    raise ValueError
            except (ValueError, TypeError):
                tree.selection_set(item)
                tree.see(item)
                messagebox.showwarning('Step Needs Duration',
                    f'Step {number} ({device}) needs a treatment duration in minutes.\n'
                    'Select it, enter the minutes, then click Update Selected Step.',
                    parent=widgets['window'])
                return
            # validate step fields
            if not str(desc).strip() or not str(device).strip():
                messagebox.showwarning('Incomplete Step',
                    f'Please complete the device and description for step {number}.',
                    parent=widgets['window'])
                return
            valid_steps.append((patient_id, number, desc, device, minutes, note))
        try:
            with get_db_connection() as con:
                con.execute('UPDATE patients SET total_prescribed_visits=? WHERE id=?',
                            (visits, patient_id))
                # replace edited plan instead of creating duplicate records
                con.execute('DELETE FROM recovery_steps WHERE patient_id=?', (patient_id,))
                con.executemany('''INSERT INTO recovery_steps
                    (patient_id,step_order,description,device_name,duration_mins,note)
                    VALUES (?,?,?,?,?,?)''', valid_steps)
                _audit_in_transaction(con, 'SAVE_PLAN', 'recovery_steps', patient_id,
                     f'{len(previous)} steps',
                     f'{len(valid_steps)} ordered steps; prescribed visits: {visits}', 'Doctor')
        except Exception as exc:
            messagebox.showerror('Save Failed', str(exc), parent=widgets['window'])
            return
        widgets['window'].destroy()
        messagebox.showinfo('Recovery Plan Saved',
            f'Saved {len(valid_steps)} ordered step(s).\n'
            f'Prescribed visits: {visits}.\n'
            'The nurse can now open Nurse Action to follow the steps.')

    # connect preset buttons + controls
    for device, button in widgets['preset_buttons'].items():
        button.configure(command=lambda d=device: quick_add(d))
    widgets['btn_add_step'].configure(command=apply_editor)
    widgets['btn_new_step'].configure(command=lambda: (reset_editor(), status(
        'Custom step: enter description, device, minutes, notes, then click Add Custom.')))
    widgets['btn_move_up'].configure(command=lambda: move_selected(-1))
    widgets['btn_move_down'].configure(command=lambda: move_selected(1))
    widgets['btn_remove_step'].configure(command=remove_selected)
    widgets['btn_save_plan'].configure(command=save_plan)
    tree.bind('<<TreeviewSelect>>', selected_editor)
    return widgets

# open execute plan window + connect buttons
def open_execute_plan_window(parent, patient_id, patient_name, main_ui):
    steps = get_recovery_steps_for_patient(patient_id)
    # warn if no plan
    if not steps:
        messagebox.showwarning('No Doctor Recovery Plan',
            'The doctor needs to save a recovery plan before the nurse starts treatment.')
        return
    widgets = build_execute_plan_window(parent, patient_name, patient_id)
    tree = widgets['tree_steps']
    visits = get_patient_visit_stats(patient_id)
    # show visit stats
    widgets['visit_label'].config(text=f"Prescribed visits: {visits['prescribed']}   |   "
        f"Attended: {visits['attended']}   |   Remaining: {visits['remaining']}")
    # load steps with pending status
    for step in steps:
        tree.insert('', tk.END, values=(step['step_order'], step['description'],
            step['device_name'], step['duration_mins'], step['note'] or '', 'Pending'))

    # get next pending step
    def next_pending():
        return next((item for item in tree.get_children()
                     if tree.item(item, 'values')[-1] != 'Done'), None)

    # show instructions for selected or next step
    def show_instructions(event=None):
        target = tree.selection()[0] if tree.selection() else next_pending()
        if target:
            order, desc, device, minutes, note, state = tree.item(target, 'values')
            widgets['instruction_label'].config(text=str(note).strip() or
                'No additional written instructions from the doctor.')
            pending = next_pending()
            # update next step banner
            if pending:
                next_step = tree.item(pending, 'values')
                widgets['next_step_label'].config(text=f'NEXT: STEP {next_step[0]} — '
                    f'{next_step[2]} ({next_step[3]} minutes)')
            else:
                widgets['next_step_label'].config(text='ALL STEPS FINISHED — Complete visit to save history')
        else:
            widgets['next_step_label'].config(text='ALL STEPS FINISHED — Complete visit to save history')
            widgets['instruction_label'].config(text='All doctor treatment steps completed.')

    # mark next pending step done
    def mark_next():
        pending = next_pending()
        # all done
        if pending is None:
            messagebox.showinfo('Completed', 'All doctor treatment steps are complete.', parent=widgets['window'])
            return
        selection = tree.selection()
        # enforce step order
        if selection and selection[0] != pending:
            tree.selection_set(pending)
            tree.see(pending)
            show_instructions()
            messagebox.showwarning('Follow the Doctor\'s Order',
                'Complete the highlighted next step. Steps cannot be skipped.', parent=widgets['window'])
            return
        values = tree.item(pending, 'values')
        tree.item(pending, values=(*values[:-1], 'Done'))
        following = next_pending()
        # select next pending step
        if following:
            tree.selection_set(following)
            tree.see(following)
        else:
            tree.selection_remove(tree.selection())
        show_instructions()

    # complete visit
    def complete_visit():
        # warn if steps unfinished
        if next_pending() is not None:
            messagebox.showwarning('Incomplete Treatment',
                'Finish every doctor-prescribed step in order before completing the visit.',
                parent=widgets['window'])
            return
        try:
            s = record_completed_visit(patient_id,
                widgets['entry_visit_note'].get('1.0', 'end-1c').strip())
        except Exception as exc:
            messagebox.showerror('Visit Not Saved', str(exc), parent=widgets['window'])
            return
        refresh_patient_queue(main_ui)
        widgets['window'].destroy()
        messagebox.showinfo('Visit Recorded',
            f"Visit recorded. Attended: {s['attended']}; Remaining: {s['remaining']}.")

    # connect handlers
    tree.bind('<<TreeviewSelect>>', show_instructions)
    widgets['btn_mark_step'].configure(command=mark_next)
    widgets['btn_complete'].configure(command=complete_visit)
    first = next_pending()
    # select first pending step
    if first is not None:
        tree.selection_set(first)
        tree.focus(first)
        tree.see(first)
    show_instructions()
    return widgets